In [ ]:
# Example 1: FileAdapter class exists and is importable.
from requests_file import FileAdapter

adapter = FileAdapter()
print(f"FileAdapter instance: {adapter}")
print(f"FileAdapter type: {type(adapter).__name__}")
print(f"FileAdapter MRO: {[c.__name__ for c in type(adapter).__mro__]}")

# It should be a requests BaseAdapter subclass
from requests.adapters import BaseAdapter
assert isinstance(adapter, BaseAdapter), "FileAdapter must subclass requests BaseAdapter"
print("FileAdapter is a requests BaseAdapter subclass")

In [ ]:
# Example 2: Mount FileAdapter on a requests.Session and fetch a local file.
import tempfile
import os
import requests
from requests_file import FileAdapter

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as tmp:
    tmp.write("hello from requests_file\n")
    tmp_path = tmp.name

try:
    session = requests.Session()
    session.mount('file://', FileAdapter())
    response = session.get(f"file://{tmp_path}")
    print(f"status_code = {response.status_code}")
    print(f"text = {response.text!r}")
    assert response.status_code == 200
    assert "hello from requests_file" in response.text
finally:
    os.unlink(tmp_path)
print("File fetch via file:// adapter succeeded")

In [ ]:
# Example 3: Fetching a nonexistent file returns 404.
import requests
from requests_file import FileAdapter

session = requests.Session()
session.mount('file://', FileAdapter())

response = session.get("file:///tmp/this-file-does-not-exist-xyz-12345.txt")
print(f"status_code for missing file = {response.status_code}")
# requests_file maps missing file to 404
assert response.status_code in (404, 400), f"Unexpected status: {response.status_code}"
print("Missing file produced expected error status")

In [ ]:
# Example 4: Binary content fetch via file:// adapter.
import tempfile
import os
import requests
from requests_file import FileAdapter

binary_payload = bytes(range(256))

with tempfile.NamedTemporaryFile(mode='wb', suffix='.bin', delete=False) as tmp:
    tmp.write(binary_payload)
    tmp_path = tmp.name

try:
    session = requests.Session()
    session.mount('file://', FileAdapter())
    response = session.get(f"file://{tmp_path}")
    assert response.status_code == 200
    assert response.content == binary_payload
    print(f"Fetched {len(response.content)} bytes of binary content")
    print(f"First 16 bytes: {response.content[:16]!r}")
finally:
    os.unlink(tmp_path)
print("Binary file fetch via file:// adapter succeeded")

In [ ]:
# Example 5: Content-Length header + percent-encoded path handling.
# Note: FileAdapter streams the file body for every method (no HEAD special-case
# upstream), so we assert the header surface on a GET instead.
import os
import tempfile
import urllib.parse
import requests
from requests_file import FileAdapter

payload = "content-length check payload\n"
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, prefix='req file ') as tmp:
    tmp.write(payload)
    tmp_path = tmp.name

try:
    session = requests.Session()
    session.mount('file://', FileAdapter())
    quoted = urllib.parse.quote(tmp_path)  # space in filename -> %20
    response = session.get(f"file://{quoted}")
    print(f"status_code = {response.status_code}")
    print(f"headers = {dict(response.headers)}")
    assert response.status_code == 200
    assert int(response.headers['Content-Length']) == os.path.getsize(tmp_path)
    assert response.text == payload
finally:
    os.unlink(tmp_path)
print("Content-Length header + percent-encoded path via file:// adapter OK")